In [2]:
import sys
from pathlib import Path 

repo_root = Path.cwd().parents[1]
if str(repo_root) not in sys.path:
    sys.path.append(str(repo_root))

In [3]:
import numpy as np
import pandas as pd
import time
import plotly.express as px
import plotly.graph_objects as go
import os
import plotly.io as pio
pio.templates.default = "plotly_white"

from pathlib import Path
from tqdm.autonotebook import tqdm
import warnings
import humanize

#If importing darts is throwing an error, import torch beforehand and then import darts
# import torch
from darts import TimeSeries
from darts.models import (
    NaiveSeasonal,
    NaiveMean,
    NaiveDrift,
    ExponentialSmoothing,
    AutoARIMA,
    ARIMA,
    Theta,
    FFT
)

from darts.metrics import mase, mse, mae, ope

from src.utils.ts_utils import forecast_bias
from src.utils.general import LogTime
from src.utils import plotting_utils
# %load_ext autoreload
# %autoreload 2
np.random.seed(42)
tqdm.pandas()

/var/folders/7v/8xmgmgf93dq9mm8nzc14n05w0000gp/T/ipykernel_37190/135939548.py:11: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm


In [4]:
os.makedirs("imgs", exist_ok=True)
preprocessed = Path('/Users/Soham/Documents/FALL2026/MTFS_textbook_work/data/london_smart_meters/preprocessed')

In [5]:
def format_plot(fig, legends = None, xlabel="Time", ylabel="Value", title="", font_size=15):
    if legends:
        names = cycle(legends)
        fig.for_each_trace(lambda t: t.update(name = next(names)))
    fig.update_layout(
        autosize=False,
        width=900,
        height=500,
        title=dict(
            text=title,
            x=0.5,
            xanchor='center',
            yanchor='top',
            font=dict(size=20),
        ),
        legend_title=None,
        legend=dict(
            font=dict(size=font_size),
            orientation="h",
            yanchor="bottom",
            y=0.98,
            xanchor="right",
            x=1,
        ),
        yaxis=dict(
            title=dict(text=ylabel, font=dict(size=font_size)),
            tickfont=dict(size=font_size),
        ),
        xaxis=dict(
            title=dict(text=xlabel, font=dict(size=font_size)),
            tickfont=dict(size=font_size),
        ),
    )
    return fig

In [6]:
try :
    train_df = pd.read_parquet(preprocessed / "selected_blocks_train_missing_imputed.parquet")
    val_df = pd.read_parquet(preprocessed / "selected_blocks_val_missing_imputed.parquet")
    test_df = pd.read_parquet(preprocessed / "selected_blocks_test_missing_imputed.parquet")
except FileNotFoundError:
    warnings.warn("The London Smart Meter dataset is not found. Please download it from https://data.london.gov.uk/dataset/smartmeter-energy-use-data-in-london-households and place it in the data/london_smart_meters directory.")

In [7]:
len(train_df.LCLid.unique())

300

In [8]:
freq = train_df.iloc[0]['frequency']
ts_train = train_df.loc[train_df.LCLid=="MAC000193", ["timestamp","energy_consumption"]].set_index("timestamp")
ts_val = val_df.loc[val_df.LCLid=="MAC000193", ["timestamp","energy_consumption"]].set_index("timestamp")
ts_test = test_df.loc[test_df.LCLid=="MAC000193", ["timestamp","energy_consumption"]].set_index("timestamp")

In [9]:
pred_df = pd.concat([ts_train, ts_val])
metric_record = []

ts_train = TimeSeries.from_dataframe(ts_train)
ts_val = TimeSeries.from_dataframe(ts_val)
ts_test = TimeSeries.from_dataframe(ts_test)

In [10]:
def eval_model(model, ts_train, ts_test, name=None):
    if name is None:
        name = type(model).__name__
    model.fit(ts_train)
    y_pred = model.predict(len(ts_test))
    return y_pred, {
        "Algorithm": name,
        "MAE": mae(actual_series = ts_test, pred_series = y_pred),
        "MSE": mse(actual_series = ts_test, pred_series = y_pred),
        "MASE": mase(actual_series = ts_test, pred_series = y_pred, insample=ts_train),
        "Forecast Bias": forecast_bias(actual_series = ts_test, pred_series = y_pred)
    }

# format_y_pred converts the darts TimeSeries object to pandas Series object.
def format_y_pred(y_pred, name):
    y_pred = y_pred.data_array().to_series() # Extract the data_array and convert to pandas Series
    y_pred.index = y_pred.index.get_level_values(0) # The darts.ts has several levels. This keeps only the index.
    y_pred.name = name # names the time-series with model's name so it can be identified in a table/legend. 
    return y_pred

In [11]:
from itertools import cycle
def plot_forecast(pred_df, forecast_columns, forecast_display_names=None):
    if forecast_display_names is None:
        forecast_display_names = forecast_columns
    else:
        assert len(forecast_columns)==len(forecast_display_names)
    mask = ~pred_df[forecast_columns[0]].isnull()
    # colors = ["rgba("+",".join([str(c) for c in plotting_utils.hex_to_rgb(c)])+",<alpha>)" for c in px.colors.qualitative.Plotly]
    colors = [c.replace("rgb", "rgba").replace(")", ", <alpha>)") for c in px.colors.qualitative.Dark2]
    # colors = [c.replace("rgb", "rgba").replace(")", ", <alpha>)") for c in px.colors.qualitative.Safe]
    act_color = colors[0]
    colors = cycle(colors[1:])
    dash_types = cycle(["dash","dot","dashdot"])
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=pred_df[mask].index, y=pred_df[mask].energy_consumption,
                        mode='lines', line = dict(color=act_color.replace("<alpha>", "0.3")),
                        name='Actual Consumption'))
    for col, display_col in zip(forecast_columns,forecast_display_names):
        fig.add_trace(go.Scatter(x=pred_df[mask].index, y=pred_df.loc[mask, col],
                            mode='lines', line = dict(dash=next(dash_types), color=next(colors).replace("<alpha>", "1")),
                            name=display_col))
    return fig

In [12]:
name = "Naive"
naive_model = NaiveSeasonal(K=1)
with LogTime() as timer:
    y_pred, metrics = eval_model(naive_model, ts_train, ts_val, name=name)
metrics['Time Elapsed'] = timer.elapsed
metric_record.append(metrics)
y_pred = format_y_pred(y_pred, "naive_predictions")
pred_df = pred_df.join(y_pred)

Time Elapsed: 4 milliseconds and 958 microseconds


In [13]:
fig = plot_forecast(pred_df, forecast_columns=["naive_predictions"], forecast_display_names=['Naive Predictions'])
fig = format_plot(fig, title=f"Naive: MAE: {metrics['MAE']:.4f} | MSE: {metrics['MSE']:.4f} | MASE: {metrics['MASE']:.4f} | Bias: {metrics['Forecast Bias']:.4f}")
fig.update_xaxes(type="date", range=["2014-01-01", "2014-01-08"])
fig.show()

In [14]:
from src.forecasting.baselines import NaiveMovingAverage

In [15]:
name = "Moving Average Forecast"
naive_model = NaiveMovingAverage(window=48)
with LogTime() as timer:
    y_pred, metrics = eval_model(naive_model, ts_train, ts_val, name=name)
metrics['Time Elapsed'] = timer.elapsed
metric_record.append(metrics)
y_pred = format_y_pred(y_pred, "moving_average_predictions")
pred_df = pred_df.join(y_pred)

Time Elapsed: 39 milliseconds and 778 microseconds


In [16]:
fig = plot_forecast(pred_df, forecast_columns=["moving_average_predictions"], forecast_display_names=['Moving Average Predictions'])
fig = format_plot(fig, title=f"Moving Average: MAE: {metrics['MAE']:.4f} | MSE: {metrics['MSE']:.4f} | MASE: {metrics['MASE']:.4f} | Bias: {metrics['Forecast Bias']:.4f}")
fig.update_xaxes(type="date", range=["2014-01-01", "2014-01-08"])
fig.show()

In [17]:
from darts.models import NaiveSeasonal 

In [18]:
name = "Seasonal Naive Forecast"
naive_model = NaiveSeasonal(K=48*7)
with LogTime() as timer:
    y_pred, metrics = eval_model(naive_model, ts_train, ts_val, name=name)
metrics['Time Elapsed'] = timer.elapsed
metric_record.append(metrics)
y_pred = format_y_pred(y_pred, "snaive_predictions")
pred_df = pred_df.join(y_pred)

Time Elapsed: 3 milliseconds and 267 microseconds


In [19]:
fig = plot_forecast(pred_df, forecast_columns=["snaive_predictions"], forecast_display_names=['Seasonal Naive Predictions'])
fig = format_plot(fig, title=f"Seasonal Naive: MAE: {metrics['MAE']:.4f} | MSE: {metrics['MSE']:.4f} | MASE: {metrics['MASE']:.4f} | Bias: {metrics['Forecast Bias']:.4f}")
fig.update_xaxes(type="date", range=["2014-01-01", "2014-01-08"])
fig.show()

In [20]:
from darts.utils.utils import ModelMode, SeasonalityMode 

In [21]:
name = "Exponential Smoothing Forecast"
with warnings.catch_warnings():
    warnings.filterwarnings("ignore", category=FutureWarning)
    ets_model = ExponentialSmoothing(trend=ModelMode.ADDITIVE, damped=True, seasonal=SeasonalityMode.ADDITIVE, seasonal_periods=48*7, random_state=42)
    with LogTime() as timer:   
        y_pred, metrics = eval_model(ets_model, ts_train, ts_val, name=name)
    metrics['Time Elapsed'] = timer.elapsed
metric_record.append(metrics)
y_pred = format_y_pred(y_pred, "ets_predictions")
pred_df = pred_df.join(y_pred)

/Users/Soham/Library/Python/3.9/lib/python/site-packages/statsmodels/tsa/holtwinters/model.py:85: RuntimeWarning: divide by zero encountered in matmul
  return err.T @ err
/Users/Soham/Library/Python/3.9/lib/python/site-packages/statsmodels/tsa/holtwinters/model.py:85: RuntimeWarning: overflow encountered in matmul
  return err.T @ err
/Users/Soham/Library/Python/3.9/lib/python/site-packages/statsmodels/tsa/holtwinters/model.py:85: RuntimeWarning: invalid value encountered in matmul
  return err.T @ err
/Users/Soham/Library/Python/3.9/lib/python/site-packages/statsmodels/tsa/holtwinters/model.py:903: ConvergenceWarning: Optimization failed to converge. Check mle_retvals.
  warnings.warn(
/Users/Soham/Library/Python/3.9/lib/python/site-packages/statsmodels/tsa/holtwinters/model.py:1377: RuntimeWarning: divide by zero encountered in matmul
  sse = err.T @ err
/Users/Soham/Library/Python/3.9/lib/python/site-packages/statsmodels/tsa/holtwinters/model.py:1377: RuntimeWarning: overflow encou

Time Elapsed: 9 seconds, 726 milliseconds and 252 microseconds


/Users/Soham/Library/Python/3.9/lib/python/site-packages/statsmodels/tsa/holtwinters/model.py:1377: RuntimeWarning: divide by zero encountered in matmul
  sse = err.T @ err
/Users/Soham/Library/Python/3.9/lib/python/site-packages/statsmodels/tsa/holtwinters/model.py:1377: RuntimeWarning: overflow encountered in matmul
  sse = err.T @ err
/Users/Soham/Library/Python/3.9/lib/python/site-packages/statsmodels/tsa/holtwinters/model.py:1377: RuntimeWarning: invalid value encountered in matmul
  sse = err.T @ err


In [22]:
fig = plot_forecast(pred_df, forecast_columns=["ets_predictions"], forecast_display_names=['Exponential Smoothing Predictions'])
fig = format_plot(fig, title=f"Exponential Smoothing: MAE: {metrics['MAE']:.4f} | MSE: {metrics['MSE']:.4f} | MASE: {metrics['MASE']:.4f} | Bias: {metrics['Forecast Bias']:.4f}")
fig.update_xaxes(type="date", range=["2014-01-01", "2014-01-08"])
fig.show()

In [23]:
name = "ARIMA"
# Not using AutoARIMA because it just takes too much time for long time series
# arima_model = AutoARIMA(max_p=5, max_q=3, m=48, seasonal=False)
arima_model = ARIMA(p=2, d=1, q=1, seasonal_order=(1, 1, 1, 48))

# Taking only latest 8000 points for training (Time constraints)
# Reduce 8000 if it is taking too much time or consuming all the memory
with LogTime() as timer:
    y_pred, metrics = eval_model(arima_model, ts_train[-8000:], ts_val, name=name)
metrics['Time Elapsed'] = timer.elapsed
metric_record.append(metrics)
y_pred = format_y_pred(y_pred, "arima_predictions")
pred_df = pred_df.join(y_pred)

/Users/Soham/Library/Python/3.9/lib/python/site-packages/numpy/linalg/_linalg.py:3220: RuntimeWarning: divide by zero encountered in matmul
  return _core_matmul(x1, x2)
/Users/Soham/Library/Python/3.9/lib/python/site-packages/numpy/linalg/_linalg.py:3220: RuntimeWarning: overflow encountered in matmul
  return _core_matmul(x1, x2)
/Users/Soham/Library/Python/3.9/lib/python/site-packages/numpy/linalg/_linalg.py:3220: RuntimeWarning: invalid value encountered in matmul
  return _core_matmul(x1, x2)


Time Elapsed: 3 minutes, 39 seconds, 363 milliseconds and 123 microseconds


In [24]:
name = "Theta"
theta_model = Theta(theta=3, seasonality_period=48*7, season_mode=SeasonalityMode.ADDITIVE)
with LogTime() as timer:
    y_pred, metrics = eval_model(theta_model, ts_train, ts_val, name=name)
metrics['Time Elapsed'] = timer.elapsed
metric_record.append(metrics)
y_pred = format_y_pred(y_pred, "theta_predictions")
pred_df = pred_df.join(y_pred)

Time Elapsed: 71 milliseconds and 484 microseconds


/Users/Soham/Library/Python/3.9/lib/python/site-packages/statsmodels/tsa/holtwinters/model.py:85: RuntimeWarning: divide by zero encountered in matmul
  return err.T @ err
/Users/Soham/Library/Python/3.9/lib/python/site-packages/statsmodels/tsa/holtwinters/model.py:85: RuntimeWarning: overflow encountered in matmul
  return err.T @ err
/Users/Soham/Library/Python/3.9/lib/python/site-packages/statsmodels/tsa/holtwinters/model.py:85: RuntimeWarning: invalid value encountered in matmul
  return err.T @ err
/Users/Soham/Library/Python/3.9/lib/python/site-packages/statsmodels/tsa/holtwinters/model.py:1377: RuntimeWarning: divide by zero encountered in matmul
  sse = err.T @ err
/Users/Soham/Library/Python/3.9/lib/python/site-packages/statsmodels/tsa/holtwinters/model.py:1377: RuntimeWarning: overflow encountered in matmul
  sse = err.T @ err
/Users/Soham/Library/Python/3.9/lib/python/site-packages/statsmodels/tsa/holtwinters/model.py:1377: RuntimeWarning: invalid value encountered in matmul

In [25]:
fig = plot_forecast(pred_df, forecast_columns=["theta_predictions"], forecast_display_names=['Theta Predictions'])
fig = format_plot(fig, title=f"Theta: MAE: {metrics['MAE']:.4f} | MSE: {metrics['MSE']:.4f} | MASE: {metrics['MASE']:.4f} | Bias: {metrics['Forecast Bias']:.4f}")
fig.update_xaxes(type="date", range=["2014-01-01", "2014-01-08"])
fig.show()

In [27]:
name = "FFT"
fft_preds = []
fft_metrics = []
lcl_ids = test_df.LCLid.unique()

for lcl_id in tqdm(lcl_ids):
    fft_model = FFT(nr_freqs_to_keep=35, trend="poly", trend_poly_degree=2)

    tr = TimeSeries.from_dataframe(
        train_df.loc[train_df.LCLid == lcl_id, ["timestamp", "energy_consumption"]].set_index("timestamp")
    )
    ts = TimeSeries.from_dataframe(
        test_df.loc[test_df.LCLid == lcl_id, ["timestamp", "energy_consumption"]].set_index("timestamp")
    )

    y_pred, metrics = eval_model(fft_model, tr, ts, name=name)
    y_pred = format_y_pred(y_pred, "predictions").to_frame()
    y_pred["LCLid"] = lcl_id
    y_pred["Algorithm"] = name
    y_pred["energy_consumption"] = ts.data_array().to_series().values

    metrics["LCLid"] = lcl_id
    metrics["Algorithm"] = name

    fft_preds.append(y_pred)
    fft_metrics.append(metrics)

fft_test_pred_df = pd.concat(fft_preds)
fft_test_metric_df = pd.DataFrame(fft_metrics)
fft_test_metric_df.head()

  0%|          | 0/300 [00:00<?, ?it/s]/Users/Soham/Library/Python/3.9/lib/python/site-packages/darts/metrics/metrics.py:384: RuntimeWarning: Mean of empty slice
  return np.nanmean(
/Users/Soham/Library/Python/3.9/lib/python/site-packages/darts/metrics/utils.py:404: RuntimeWarning: Mean of empty slice
  vals = component_reduction(vals, axis=COMP_AX)
/Users/Soham/Library/Python/3.9/lib/python/site-packages/darts/metrics/metrics.py:797: RuntimeWarning: Mean of empty slice
  return np.nanmean(
/Users/Soham/Library/Python/3.9/lib/python/site-packages/darts/metrics/metrics.py:616: RuntimeWarning: Mean of empty slice
  return np.nanmean(
/Users/Soham/Documents/FALL2026/MTFS_textbook_work/src/utils/ts_utils.py:104: RuntimeWarning: invalid value encountered in scalar divide
  return ((y_true_sum - y_pred_sum) / y_true_sum) * 100.
  0%|          | 1/300 [00:00<02:47,  1.78it/s]/Users/Soham/Library/Python/3.9/lib/python/site-packages/darts/metrics/metrics.py:384: RuntimeWarning: Mean of empty s

,Algorithm,MAE,MSE,MASE,Forecast Bias,LCLid
0,FFT,NaN,NaN,NaN,NaN,MAC003449
1,FFT,NaN,NaN,NaN,NaN,MAC000768
2,FFT,NaN,NaN,NaN,NaN,MAC000948
3,FFT,NaN,NaN,NaN,NaN,MAC000974
4,FFT,NaN,NaN,NaN,NaN,MAC001734


In [28]:
# FFT for MAC000193 on the validation period, so it lines up with the Theta predictions
fft_single_model = FFT(nr_freqs_to_keep=35, trend="poly", trend_poly_degree=2)
y_pred_fft, _ = eval_model(fft_single_model, ts_train, ts_val, name="FFT")
y_pred_fft = format_y_pred(y_pred_fft, "fft_predictions")
pred_df = pred_df.join(y_pred_fft)

fig = plot_forecast(pred_df,
                    forecast_columns=["theta_predictions", "fft_predictions"],
                    forecast_display_names=["Theta", "FFT"])
fig = format_plot(fig, title="Theta and FFT Forecasts")
fig.update_xaxes(type="date", range=["2014-01-01", "2014-01-08"])
fig.show()